In [1]:
# Load packages 
import anndata as ad 
import pandas as pd 
import numpy as np

In [2]:
# To-do:
# Make sure num_junctions column exists 
# Make sure junction counts and ATSE counts consistent 
# RuntimeError: indices and values must have same nnz, but got nnz from indices: 10906620, nnz from values: 11613541

In [3]:
# Paths 
orthologous_junctions_path = "/gpfs/commons/home/kisaev/Leaflet-analysis/Multi_Species_Splicing_Foundation/plots_2025-08-14/junction_mapping_mouse_human_with_annotations.csv"
orthologous_junctions = pd.read_csv(orthologous_junctions_path)

# Load mouse anndata 
mouse_anndata_file = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/MOUSE_SPLICING_FOUNDATION/MODEL_INPUT/072025/aligned_splicing_data_20250730_164104.h5ad"
mouse = ad.read_h5ad(mouse_anndata_file)

# Load human anndata 
human_anndata_file = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/HUMAN_SPLICING_FOUNDATION/MODEL_INPUT/072025/aligned_splicing_data_20250731_212313.h5ad"
human = ad.read_h5ad(human_anndata_file)


In [4]:
mouse_present = orthologous_junctions["mouse_junction_id"].isin(mouse.var["junction_id"])
human_present = orthologous_junctions["human_junction_id"].isin(human.var["junction_id"])

# Subset orthologous junctions to only include junctions that are present in mouse and human
orthologous_junctions = orthologous_junctions[mouse_present & human_present]

# Filter mouse anndata for orthologous junctions
mouse = mouse[:, mouse.var["junction_id"].isin(orthologous_junctions["mouse_junction_id"])].copy()

# Filter human anndata for orthologous junctions
human = human[:, human.var["junction_id"].isin(orthologous_junctions["human_junction_id"])].copy()

# Create universal junction id hash for orthologous junctions
orthologous_junctions["joint_junction_id"] = orthologous_junctions["mouse_junction_id"] + "_" + orthologous_junctions["human_junction_id"]

human_ortho = orthologous_junctions[["human_junction_id", "joint_junction_id"]]
human_ortho.columns = ["junction_id", "joint_junction_id"]

mouse_ortho = orthologous_junctions[["mouse_junction_id", "joint_junction_id"]]
mouse_ortho.columns = ["junction_id", "joint_junction_id"]

# Merge mouse.var and human.var with orthologous_junctions
mouse.var = mouse.var.merge(mouse_ortho, on="junction_id", how="left")
human.var = human.var.merge(human_ortho, on="junction_id", how="left")

# Order mouse and human anndatas by joint junction id
mouse.var = mouse.var.sort_values(by="joint_junction_id")
human.var = human.var.sort_values(by="joint_junction_id")

# Assert that order of joint junction id is the same in mouse and human anndatas
assert np.all(mouse.var["joint_junction_id"].values == human.var["joint_junction_id"].values), "ERROR: Order of joint junction id is not the same in mouse and human anndatas!"


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/anndata.py:750: UserWarning: 
AnnData expects .var.index to contain strings, but got values like:
    [2291, 2290, 2289, 2288, 2287]

    Inferred to be: integer

  value_idx = self._prep_dim_index(value.index, attr)


In [5]:
# Now we gotta clean up the .obs to ensure similar columns 
mouse.obs["species"] = "mouse"
human.obs["species"] = "human"

# Standardize .obs columns for concatenation
print("\nStandardizing .obs columns...")

# Columns to drop
cols_to_drop = ['AgingScore_unweighted', 'AgingScore_pos', 'AgingScore_neg']

# Drop columns from mouse.obs if they exist
mouse_cols_to_drop = [col for col in cols_to_drop if col in mouse.obs.columns]
if mouse_cols_to_drop:
    mouse.obs = mouse.obs.drop(columns=mouse_cols_to_drop)
    print(f"Dropped from mouse: {mouse_cols_to_drop}")

# Drop columns from human.obs if they exist
human_cols_to_drop = [col for col in cols_to_drop if col in human.obs.columns]
if human_cols_to_drop:
    human.obs = human.obs.drop(columns=human_cols_to_drop)
    print(f"Dropped from human: {human_cols_to_drop}")

# Rename columns for consistency
mouse_rename_dict = {
    'cell_ontology_class': 'cell_type',
    'mouse.id': 'donor'
}
mouse.obs = mouse.obs.rename(columns=mouse_rename_dict)
print(f"Renamed in mouse: {mouse_rename_dict}")

human_rename_dict = {
    'cell_id_clean': 'cell_clean'
}
human.obs = human.obs.rename(columns=human_rename_dict)
print(f"Renamed in human: {human_rename_dict}")

print("\n.obs standardization complete.")

# Display cleaned columns for verification
print("\nMouse OBS columns after cleaning:")
print(mouse.obs.columns.tolist())

print("\nHuman OBS columns after cleaning:")
print(human.obs.columns.tolist())

# Find common columns for verification
common_cols = list(set(mouse.obs.columns) & set(human.obs.columns))
print(f"\nFound {len(common_cols)} common columns between mouse and human:")
print(sorted(common_cols))

# Columns unique to mouse
mouse_unique = list(set(mouse.obs.columns) - set(human.obs.columns))
print(f"\n{len(mouse_unique)} columns unique to mouse:")
print(sorted(mouse_unique))

# Columns unique to human
human_unique = list(set(human.obs.columns) - set(mouse.obs.columns))
print(f"\n{len(human_unique)} columns unique to human:")
print(sorted(human_unique))

# Subset mouse and human anndatas to only include common columns
mouse.obs = mouse.obs[common_cols]
human.obs = human.obs[common_cols]

# Make sure the columns in both .obs are the same order 
assert np.all(mouse.obs.columns.values == human.obs.columns.values), "ERROR: Columns in .obs are not the same order in mouse and human anndatas!"

# Convert mouse age to numeric values   
mouse.obs["age"] = mouse.obs["age"].str.replace("m", "").astype(float)
mouse.obs["age_units"] = "months"
human.obs["age_units"] = "years"


Standardizing .obs columns...
Dropped from mouse: ['AgingScore_unweighted', 'AgingScore_pos', 'AgingScore_neg']
Dropped from human: ['AgingScore_unweighted', 'AgingScore_pos', 'AgingScore_neg']
Renamed in mouse: {'cell_ontology_class': 'cell_type', 'mouse.id': 'donor'}
Renamed in human: {'cell_id_clean': 'cell_clean'}

.obs standardization complete.

Mouse OBS columns after cleaning:
['cell_id_index', 'age', 'cell_type', 'donor', 'sex', 'subtissue', 'tissue', 'dataset', 'cell_name', 'cell_id', 'cell_clean', 'specific_cell_type', 'broad_cell_type', 'medium_cell_type', 'seqtech', 'library_size', 'total_junction_reads', 'annotated_junction_reads', 'unannotated_junction_reads', 'n_detected_annotated_junctions', 'n_detected_unannotated_junctions', 'nuclear_ratio_log_norm', 'species']

Human OBS columns after cleaning:
['cell_id', 'donor', 'sex', 'age', 'dataset', 'tissue', 'cell_type', 'broad_cell_type', 'cell_id_index', 'cell_clean', 'total_junction_reads', 'annotated_junction_reads', 'un

In [6]:
# find common columns between mouse.var and human.var 
common_var_cols = list(set(mouse.var.columns) & set(human.var.columns))
print(f"\nFound {len(common_var_cols)} common columns between mouse and human:")
print(sorted(common_var_cols))

mouse.var = mouse.var[common_var_cols]
human.var = human.var[common_var_cols]


Found 14 common columns between mouse and human:
['CountJuncs', 'annotation_status', 'confidence', 'event_id', 'gene_id', 'gene_name', 'joint_junction_id', 'junction_id', 'junction_id_index', 'n_cells_detected', 'num_junctions', 'position_off_3_prime', 'position_off_5_prime', 'splice_motif']


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/anndata.py:750: UserWarning: 
AnnData expects .var.index to contain strings, but got values like:
    [2291, 2290, 2289, 2288, 2287]

    Inferred to be: integer

  value_idx = self._prep_dim_index(value.index, attr)


In [23]:
human.var = mouse.var

In [29]:
# Try concatenating mouse and human anndatas    
combined = ad.concat([mouse, human], axis=0, join='outer', 
                     index_unique='-', fill_value=0)

combined.var = mouse.var

print(f"Combined shape: {combined.shape}")
print(f"Mouse cells: {sum(combined.obs['species'] == 'mouse')}")
print(f"Human cells: {sum(combined.obs['species'] == 'human')}")
print(f"Layers: {list(combined.layers.keys())}")

Combined shape: (213259, 14237)
Mouse cells: 137936
Human cells: 75323
Layers: ['cell_by_cluster_matrix', 'cell_by_junction_matrix']


/gpfs/commons/home/kisaev/miniconda3/envs/LeafletSC/lib/python3.10/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [30]:
combined.obs["cell_id_index"] = np.arange(combined.shape[0])
combined.var["junction_id_index"] = np.arange(combined.shape[1])

In [34]:
combined.obs.reset_index(drop=True, inplace=True)

In [36]:
# clean up combined.obs["sex"] so always F and M if female and male respectively change to F and M 
combined.obs["sex"] = combined.obs["sex"].str.replace("female", "F")
combined.obs["sex"] = combined.obs["sex"].str.replace("male", "M")
print(combined.obs["sex"].unique())

['F' 'M']


In [39]:
# Mapping dictionary for tissue cleanup
tissue_mapping = {
    # Brain
    "Brain_Non-Myeloid": "Brain",
    "Brain_Myeloid": "Brain",
    "MTG": "Brain_Cortex",
    "V1C": "Brain_Cortex",
    "A1C": "Brain_Cortex",
    "CgG": "Brain_Cortex",
    "M1lm": "Brain_Cortex",
    "S1ul": "Brain_Cortex",
    "S1lm": "Brain_Cortex",
    "M1ul": "Brain_Cortex",

    # Immune / Hematopoietic
    "Marrow": "Bone_Marrow",
    "Bone_Marrow": "Bone_Marrow",
    "Blood": "Blood",
    "Spleen": "Spleen",
    "Thymus": "Thymus",
    "Lymph_Node": "Lymph_Node",

    # Heart & Muscle
    "Heart": "Heart",
    "Muscle": "Skeletal_Muscle",
    "Limb_Muscle": "Skeletal_Muscle",
    "Diaphragm": "Skeletal_Muscle",
    "Tongue": "Tongue",

    # Adipose
    "SCAT": "Adipose_Tissue",
    "GAT": "Adipose_Tissue",
    "MAT": "Adipose_Tissue",
    "BAT": "Adipose_Tissue",
    "Fat": "Adipose_Tissue",

    # Digestive
    "Large_Intestine": "Large_Intestine",
    "Small_Intestine": "Small_Intestine",
    "Pancreas": "Pancreas",
    "Salivary_Gland": "Salivary_Gland",
    "Liver": "Liver",

    # Lung & Airways
    "Lung": "Lung",
    "Trachea": "Trachea",

    # Skin & Related
    "Skin": "Skin",
    "Mammary_Gland": "Mammary_Gland",
    "Mammary": "Mammary_Gland",

    # Kidney & Urinary
    "Kidney": "Kidney",
    "Bladder": "Bladder",

    # Reproductive
    "Uterus": "Uterus",
    "Prostate": "Prostate",

    # Vasculature / Connective
    "Vasculature": "Vasculature",
    "Aorta": "Aorta",

    # Special
    "Eye": "Eye"
}

# Apply mapping
combined.obs["tissue_clean"] = combined.obs["tissue"].map(tissue_mapping).fillna(combined.obs["tissue"])
print(combined.obs["tissue_clean"].value_counts())

tissue_clean
Brain              66428
Brain_Cortex       44976
Bone_Marrow        14002
Adipose_Tissue     11075
Skeletal_Muscle     9122
Heart               8145
Large_Intestine     7026
Lung                6257
Skin                6036
Spleen              5361
Thymus              4710
Tongue              4194
Bladder             3136
Trachea             2894
Lymph_Node          2845
Liver               2716
Pancreas            2414
Mammary_Gland       2095
Kidney              1870
Vasculature         1847
Blood               1746
Small_Intestine     1474
Salivary_Gland      1198
Aorta                640
Uterus               423
Eye                  323
Prostate             306
Name: count, dtype: int64


In [44]:
broad_cell_mapping = {
    # Neurons
    "Neuron": "Neuron",
    "Excitatory_Neuron": "Neuron",
    "Inhibitory_Neuron": "Neuron",
    "Other_Neuron": "Neuron",

    # Glia
    "Glial cell": "Glia",
    "CNS_Glia": "Glia",
    "PNS_Glia": "Glia",
    "Microglia": "Glia",

    # Immune
    "Immune cell": "Immune",
    "Immune_Other": "Immune",
    "B_cell": "B cell lineage",
    "Plasma_cell": "B cell lineage",
    "CD4_T_cell": "T cell",
    "CD8_T_cell": "T cell",
    "Other_T_cell": "T cell",
    "Regulatory_T_cell": "T cell",
    "NK_ILC": "NK/ILC",
    "Monocyte": "Myeloid",
    "Macrophage": "Myeloid",
    "Dendritic_cell": "Myeloid",
    "Granulocyte": "Myeloid",
    "Myeloid_Other": "Myeloid",
    "Hematopoietic_Progenitor": "Hematopoietic progenitor",
    "Hematopoietic_Mature": "Hematopoietic progenitor",
    "Stem_Progenitor_Other": "Hematopoietic progenitor",

    # Epithelial
    "Epithelial cell": "Epithelial",
    "GI_Epithelial": "Epithelial",
    "Intestinal cell": "Epithelial",
    "Other_Epithelial": "Epithelial",
    "Urogenital_Epithelial": "Epithelial",
    "Respiratory_Epithelial": "Epithelial",
    "Alveolar_cell": "Epithelial",
    "Secretory cell": "Secretory epithelial",
    "Secretory_Gland": "Secretory epithelial",

    # Fibroblast / Stromal
    "Stromal cell": "Fibroblast",
    "General_Fibroblast": "Fibroblast",
    "Organ_Specific_Fibroblast": "Fibroblast",
    "Mesenchymal_Stem": "Mesenchymal stem/stromal",

    # Muscle
    "Muscle cell": "Muscle",
    "Skeletal_Muscle": "Muscle",
    "Cardiac cell": "Muscle",
    "Smooth_Muscle": "Muscle",
    "Bladder cell": "Muscle",
    "Muscle_Other": "Muscle",

    # Vascular / Endothelial / Perivascular
    "Vascular cell": "Endothelial",
    "Endothelial": "Endothelial",
    "Capillary_Endothelial": "Endothelial",
    "Venous_Endothelial": "Endothelial",
    "Arterial_Endothelial": "Endothelial",
    "Lymphatic_Endothelial": "Endothelial",
    "Specialized_Endothelial": "Endothelial",
    "Fenestrated cell": "Endothelial",
    "Pericyte": "Pericyte",

    # Stem/Progenitor
    "Stem cell": "Stem/Progenitor",
    "Neuroepithelial cell": "Stem/Progenitor",
    "Stem_Progenitor_Other": "Stem/Progenitor",

    # Organ-specific (optional grouping, otherwise leave distinct)
    "Pancreatic cell": "Organ-specific epithelial",
    "Liver cell": "Organ-specific epithelial",
    "Kidney cell": "Organ-specific epithelial",
    "Lung cell": "Organ-specific epithelial",

    # Catch-all
    "Other": "Other/Unclassified",
    "Other cell": "Other/Unclassified",
}

# Apply mapping
combined.obs["broad_cell_type_clean"] = combined.obs["broad_cell_type"].map(broad_cell_mapping).fillna(combined.obs["broad_cell_type"])


In [48]:
combined.layers["cell_by_junction_matrix"]

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 298811512 stored elements and shape (213259, 14237)>

In [50]:
combined.layers["cell_by_cluster_matrix"]

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 298811512 stored elements and shape (213259, 14237)>

In [51]:
# Save into anndata file
output_file = "/gpfs/commons/groups/knowles_lab/Karin/Leaflet-analysis-WD/CROSS_SPECIES_AGING/Leaflet/input_files/joint_anndata_20250903.h5ad"
combined.write_h5ad(output_file)